# Predictive Maintenance: Equipment Failure Analysis

## Objective

The objective of this project is to investigate industrial
machine sensor data and develop a machine-learning approach
for identifying equipment failures.

## Dataset

Source: UCI Machine Learning Repository

Dataset: AI4I 2020 Predictive Maintenance Dataset

## Initial Questions

1. What information does the dataset contain?
2. Are there missing or duplicated records?
3. How frequently do equipment failures occur?
4. Which sensor measurements might be associated with failures?
5. How well can a machine-learning model identify failures?

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
import pandas as pd
from pathlib import Path

# Load the dataset
data_path = Path("../data/raw/ai4i2020.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (10000, 14)


In [5]:
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

# Define the dataset path
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"

# Load the dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [7]:
df.isnull().sum()


UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
UDI,10000.0,5000.50000,2886.895680,1.0,2500.75,5000.5,7500.25,10000.0
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.30,300.1,301.50,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.80,310.1,311.10,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.00,1503.0,1612.00,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.20,40.1,46.80,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.00,108.0,162.00,253.0
Machine failure,10000.0,0.03390,0.180981,0.0,0.00,0.0,0.00,1.0
TWF,10000.0,0.00460,0.067671,0.0,0.00,0.0,0.00,1.0
HDF,10000.0,0.01150,0.106625,0.0,0.00,0.0,0.00,1.0
PWF,10000.0,0.00950,0.097009,0.0,0.00,0.0,0.00,1.0


## 1. Understanding the Dataset

The AI4I 2020 Predictive Maintenance Dataset contains synthetic industrial machine data.

The dataset includes machine identifiers, product information, sensor measurements, and equipment failure indicators.

The purpose of this analysis is to understand the available features and identify the variables relevant to predictive maintenance.


In [10]:
# Display all column names
print("Dataset columns:\n")

for column in df.columns:
    print(column)

Dataset columns:

UDI
Product ID
Type
Air temperature [K]
Process temperature [K]
Rotational speed [rpm]
Torque [Nm]
Tool wear [min]
Machine failure
TWF
HDF
PWF
OSF
RNF


In [8]:
failure_counts = df["Machine failure"].value_counts().sort_index()

failure_summary = pd.DataFrame({
    "Count": failure_counts,
    "Percentage": (
        failure_counts / len(df) * 100
    ).round(2)
})

failure_summary.index = [
    "No Failure" if i == 0 else "Failure"
    for i in failure_summary.index
]

failure_summary

,Count,Percentage
No Failure,9661,96.61
Failure,339,3.39


In [9]:
print("Machine type counts:")
print(df["Type"].value_counts())

print("\nMachine type percentages:")
print(
    (df["Type"].value_counts(normalize=True) * 100).round(2)
)

Machine type counts:
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

Machine type percentages:
Type
L    60.00
M    29.97
H    10.03
Name: proportion, dtype: float64


In [10]:
columns_to_review = [
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

df[columns_to_review].head()

,UDI,Product ID,TWF,HDF,PWF,OSF,RNF
0,1,M14860,0,0,0,0,0
1,2,L47181,0,0,0,0,0
2,3,L47182,0,0,0,0,0
3,4,L47183,0,0,0,0,0
4,5,L47184,0,0,0,0,0
